In [3]:
#!/usr/bin/env python3
"""
A-to-Z Stock Analyzer — India-first, multi-strategy edition
===========================================================

Features
--------
- NSE/BSE/US ticker resolution (Yahoo Finance symbols)
- Company overview, valuation, profitability, growth, debt, liquidity,
  cash flow, dividends, analyst estimates and ownership where available
- Annual and quarterly financial trend snapshots
- Technical indicators: SMA/EMA, RSI, MACD, Bollinger Bands, ATR, ADX,
  volume, support/resistance, 52-week high/low
- Separate strategy plans: short-term, swing, positional and long-term
- Entry zone, breakout trigger, stop loss, targets, risk/reward, quantity
- Position size based on configured capital and risk limit
- Recent news headline sentiment (simple keyword heuristic)
- PNG chart and JSON report output
- Missing/unavailable data is reported as N/A, not silently treated as good

Install:
    python -m pip install yfinance pandas numpy matplotlib

Run:
    python stock_analyzer.py
    python stock_analyzer.py RELIANCE
    python stock_analyzer.py "Tata Motors"
    python stock_analyzer.py AAPL

Yahoo Finance data can be delayed, incomplete, or incorrect. Verify prices and
financials against exchange filings and company reports before trading.
This is an educational research aid, not financial advice or a prediction.
"""

from __future__ import annotations

import json
import math
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------- SETTINGS -----------------------------
CAPITAL = 100_000.0             # INR for Indian stocks; change for your account
RISK_PER_TRADE_PCT = 1.0        # maximum planned loss as % of capital
MAX_CAPITAL_PER_TRADE_PCT = 25.0
SAVE_CHART = True
SAVE_JSON = True
OUTPUT_DIR = Path("stock_reports")

# Strategy settings: daily bars except long-term uses weekly bars.
STRATEGIES = {
    "SHORT-TERM (1–10 trading days)": {
        "period": "6mo", "interval": "1d", "atr_mult": 1.5,
        "support_window": 10, "resistance_window": 10, "rr": [1.5, 2.0, 3.0],
        "min_bars": 40,
    },
    "SWING TRADE (2–8 weeks)": {
        "period": "1y", "interval": "1d", "atr_mult": 2.0,
        "support_window": 20, "resistance_window": 20, "rr": [1.5, 2.5, 3.5],
        "min_bars": 60,
    },
    "POSITIONAL (2–6 months)": {
        "period": "2y", "interval": "1d", "atr_mult": 2.5,
        "support_window": 60, "resistance_window": 60, "rr": [2.0, 3.0, 4.0],
        "min_bars": 120,
    },
    "LONG-TERM (1–5+ years)": {
        "period": "5y", "interval": "1wk", "atr_mult": 3.0,
        "support_window": 26, "resistance_window": 26, "rr": [2.0, 3.0, 5.0],
        "min_bars": 52,
    },
}

POSITIVE_WORDS = {
    "surge", "jump", "rally", "gain", "profit", "beat", "record", "growth",
    "upgrade", "buy", "strong", "wins", "order", "expansion", "dividend",
    "bullish", "outperform", "soars", "rises", "approval", "contract",
}
NEGATIVE_WORDS = {
    "fall", "drop", "plunge", "loss", "miss", "decline", "downgrade", "sell",
    "weak", "probe", "fraud", "penalty", "lawsuit", "cut", "bearish",
    "underperform", "slump", "crash", "debt", "fine", "default", "resign",
}

# ------------------------------- HELPERS -------------------------------
def safe_float(value: Any) -> float | None:
    try:
        if value is None or pd.isna(value):
            return None
        result = float(value)
        return result if math.isfinite(result) else None
    except (TypeError, ValueError):
        return None


def fmt(value: Any, digits: int = 2, suffix: str = "") -> str:
    value = safe_float(value)
    return "N/A" if value is None else f"{value:,.{digits}f}{suffix}"


def percent(value: Any, digits: int = 1) -> str:
    value = safe_float(value)
    return "N/A" if value is None else f"{value * 100:.{digits}f}%"


def money(value: Any, currency: str = "INR") -> str:
    value = safe_float(value)
    if value is None:
        return "N/A"
    if currency == "INR":
        absolute = abs(value)
        if absolute >= 1e7:
            return f"{'-' if value < 0 else ''}₹{absolute / 1e7:,.2f} Cr"
        if absolute >= 1e5:
            return f"{'-' if value < 0 else ''}₹{absolute / 1e5:,.2f} lakh"
        return f"₹{value:,.2f}"
    return f"{value:,.2f} {currency}"


def section(title: str) -> None:
    print("\n" + "=" * 88)
    print(f" {title}")
    print("=" * 88)


def clean_for_json(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return safe_float(obj)
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    if pd.isna(obj) if not isinstance(obj, (dict, list, tuple)) else False:
        return None
    return obj


def resolve_symbol(user_input: str) -> str | None:
    raw = user_input.strip()
    if not raw:
        return None
    upper = raw.upper().strip()
    # Preserve explicit Yahoo symbols and index tickers.
    if "." in upper or upper.startswith("^") or "=" in upper:
        candidates = [upper]
    else:
        compact = re.sub(r"\s+", "", upper)
        candidates = [compact + ".NS", compact + ".BO", compact]
    for candidate in candidates:
        try:
            hist = yf.Ticker(candidate).history(period="5d", interval="1d", auto_adjust=True)
            if hist is not None and not hist.empty:
                return candidate
        except Exception:
            continue
    try:
        search = yf.Search(raw, max_results=10)
        quotes = getattr(search, "quotes", []) or []
        # Prefer NSE, then BSE; otherwise first plausible result.
        for suffix in (".NS", ".BO"):
            for quote in quotes:
                symbol = quote.get("symbol", "")
                if symbol.endswith(suffix):
                    return symbol
        if quotes:
            return quotes[0].get("symbol")
    except Exception:
        pass
    return None


def score_rows(rows: list[dict]) -> float:
    """Normalize criteria scored between -max_points and +max_points to 0–100."""
    total_max = sum(float(row["max_points"]) for row in rows)
    if not rows or total_max <= 0:
        return 50.0
    total = sum(float(row["points"]) for row in rows)
    return round((total + total_max) / (2 * total_max) * 100, 1)


def print_rows(rows: list[dict]) -> None:
    if not rows:
        print("  Not enough data to score this section.")
        return
    for row in rows:
        print(f"  {row['metric']:<28} {row['value']:<16} {row['assessment']}")


def get_info(ticker: yf.Ticker) -> dict:
    try:
        result = ticker.get_info()
        return result if isinstance(result, dict) else {}
    except Exception:
        try:
            result = ticker.info
            return result if isinstance(result, dict) else {}
        except Exception:
            return {}


def statement_value(frame: pd.DataFrame | None, possible_names: list[str], column: Any) -> float | None:
    if frame is None or frame.empty:
        return None
    for name in possible_names:
        if name in frame.index:
            return safe_float(frame.loc[name, column])
    return None


# --------------------------- FUNDAMENTALS ---------------------------
def analyze_fundamentals(info: dict) -> tuple[list[dict], float]:
    rows: list[dict] = []

    def add(metric: str, value: Any, points: float, max_points: float, assessment: str) -> None:
        rows.append({
            "metric": metric,
            "value": str(value),
            "points": points,
            "max_points": max_points,
            "assessment": assessment,
        })

    sector = str(info.get("sector") or "")
    financial = any(word in sector.lower() for word in ("financial", "bank", "insurance"))
    pe = safe_float(info.get("trailingPE"))
    if pe is not None:
        if pe <= 0:
            add("Trailing P/E", fmt(pe), -2, 2, "Loss-making / P/E not meaningful")
        elif pe < 15:
            add("Trailing P/E", fmt(pe), 2, 2, "Lower valuation; compare with peers")
        elif pe < 25:
            add("Trailing P/E", fmt(pe), 1, 2, "Moderate; compare with peers")
        elif pe < 40:
            add("Trailing P/E", fmt(pe), 0, 2, "Premium valuation")
        else:
            add("Trailing P/E", fmt(pe), -1, 2, "High expectations priced in")

    fpe = safe_float(info.get("forwardPE"))
    if fpe is not None and pe is not None and pe > 0 and fpe > 0:
        if fpe < pe * 0.9:
            add("Forward P/E", fmt(fpe), 1, 1, "Market expects earnings improvement")
        elif fpe > pe * 1.1:
            add("Forward P/E", fmt(fpe), -1, 1, "Forward valuation is higher")

    peg = safe_float(info.get("trailingPegRatio") or info.get("pegRatio"))
    if peg is not None and peg > 0:
        pts, note = (2, "Potentially reasonable growth valuation") if peg < 1 else (
            (0, "Moderate PEG") if peg < 2 else (-1, "Expensive versus estimated growth")
        )
        add("PEG ratio", fmt(peg), pts, 2, note)

    pb = safe_float(info.get("priceToBook"))
    if pb is not None and pb > 0:
        if financial:
            pts, note = ((1, "Assess with ROE and asset quality") if pb < 3 else (-1, "High P/B; check ROE and asset quality"))
        else:
            pts, note = ((1, "Lower P/B; industry context matters") if pb < 3 else (-1, "High P/B; check growth and ROCE"))
        add("Price / Book", fmt(pb), pts, 1, note)

    for key, label, good, fair in [
        ("returnOnEquity", "ROE", 0.15, 0.08),
        ("returnOnAssets", "ROA", 0.08, 0.03),
        ("profitMargins", "Net profit margin", 0.15, 0.08),
        ("operatingMargins", "Operating margin", 0.20, 0.10),
    ]:
        value = safe_float(info.get(key))
        if value is None or (label == "ROA" and not financial and value < -1):
            continue
        if label == "ROA" and not financial:
            # ROA is particularly sector-sensitive; show it without over-weighting.
            maxp = 1
        else:
            maxp = 2
        if value >= good:
            add(label, percent(value), maxp, maxp, "Strong")
        elif value >= fair:
            add(label, percent(value), 1, maxp, "Moderate")
        elif value >= 0:
            add(label, percent(value), 0, maxp, "Weak / needs context")
        else:
            add(label, percent(value), -maxp, maxp, "Negative")

    for key, label in [("revenueGrowth", "Revenue growth (reported period)"),
                       ("earningsGrowth", "Earnings growth (reported period)")]:
        value = safe_float(info.get(key))
        if value is None:
            continue
        if value > 0.15:
            add(label, percent(value), 2, 2, "Strong growth")
        elif value > 0.05:
            add(label, percent(value), 1, 2, "Moderate growth")
        elif value >= 0:
            add(label, percent(value), 0, 2, "Low / flat growth")
        else:
            add(label, percent(value), -2, 2, "Declining")

    debt_equity = safe_float(info.get("debtToEquity"))
    if debt_equity is not None and not financial:
        ratio = debt_equity / 100.0  # Yahoo commonly reports this as a percentage
        if ratio < 0.5:
            add("Debt / equity", fmt(ratio), 2, 2, "Low debt")
        elif ratio < 1:
            add("Debt / equity", fmt(ratio), 1, 2, "Manageable; inspect cash flow")
        elif ratio < 2:
            add("Debt / equity", fmt(ratio), -1, 2, "High debt")
        else:
            add("Debt / equity", fmt(ratio), -2, 2, "Very high debt")

    current_ratio = safe_float(info.get("currentRatio"))
    if current_ratio is not None and not financial:
        if current_ratio >= 1.5:
            add("Current ratio", fmt(current_ratio), 1, 1, "Comfortable short-term liquidity")
        elif current_ratio < 1:
            add("Current ratio", fmt(current_ratio), -1, 1, "Potential liquidity pressure")

    fcf = safe_float(info.get("freeCashflow"))
    if fcf is not None:
        add("Free cash flow", money(fcf, info.get("financialCurrency") or info.get("currency") or ""),
            1 if fcf > 0 else -1, 1, "Positive cash generation" if fcf > 0 else "Negative cash flow")

    div_yield = safe_float(info.get("dividendYield"))
    if div_yield is not None:
        add("Dividend yield", percent(div_yield), 1 if div_yield > 0.02 else 0, 1,
            "Meaningful yield" if div_yield > 0.02 else "Low / no yield; not necessarily negative")

    target = safe_float(info.get("targetMeanPrice"))
    current = safe_float(info.get("currentPrice") or info.get("regularMarketPrice"))
    if target is not None and current and current > 0:
        upside = target / current - 1
        pts = 1 if upside > 0.15 else (-1 if upside < -0.05 else 0)
        add("Analyst mean target upside", percent(upside), pts, 1,
            f"Consensus estimate; {info.get('numberOfAnalystOpinions', 'unknown')} analysts")

    return rows, score_rows(rows)


def print_financial_trends(ticker: yf.Ticker, currency: str) -> dict:
    trends: dict[str, list[dict]] = {}
    for label, attr, names in [
        ("Annual", "income_stmt", ["Total Revenue", "Net Income", "Operating Income", "Diluted EPS"]),
        ("Quarterly", "quarterly_income_stmt", ["Total Revenue", "Net Income", "Operating Income", "Diluted EPS"]),
    ]:
        try:
            frame = getattr(ticker, attr)
        except Exception:
            frame = None
        if frame is None or frame.empty:
            continue
        records = []
        for col in list(frame.columns)[:5]:
            item = {"period": str(pd.Timestamp(col).date()) if not isinstance(col, str) else col}
            for name in names:
                val = statement_value(frame, [name], col)
                item[name] = val
            records.append(item)
        trends[label] = records
        print(f"\n  {label} financial snapshot (most recent first):")
        for item in records:
            rev = money(item.get("Total Revenue"), currency)
            ni = money(item.get("Net Income"), currency)
            op = money(item.get("Operating Income"), currency)
            eps = fmt(item.get("Diluted EPS"))
            print(f"    {item['period']}: Revenue {rev} | Net profit {ni} | Operating profit {op} | Diluted EPS {eps}")
    return trends


# ----------------------------- TECHNICALS -----------------------------
def compute_indicators(history: pd.DataFrame) -> pd.DataFrame:
    d = history.copy()
    # yfinance sometimes returns multi-index columns for a single ticker.
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    for col in ("Close", "High", "Low", "Volume"):
        if col not in d.columns:
            raise ValueError(f"Price history is missing required column: {col}")

    c, h, low, volume = d["Close"], d["High"], d["Low"], d["Volume"]
    for n in (10, 20, 50, 100, 200):
        d[f"SMA{n}"] = c.rolling(n).mean()
    for n in (9, 20, 50):
        d[f"EMA{n}"] = c.ewm(span=n, adjust=False).mean()

    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    avg_loss = loss.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    d["RSI"] = 100 - (100 / (1 + rs))

    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d["MACD"] = ema12 - ema26
    d["MACD_SIGNAL"] = d["MACD"].ewm(span=9, adjust=False).mean()
    d["MACD_HIST"] = d["MACD"] - d["MACD_SIGNAL"]

    previous_close = c.shift(1)
    true_range = pd.concat([
        h - low, (h - previous_close).abs(), (low - previous_close).abs()
    ], axis=1).max(axis=1)
    d["ATR"] = true_range.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()

    std = c.rolling(20).std()
    d["BB_UPPER"] = d["SMA20"] + 2 * std
    d["BB_LOWER"] = d["SMA20"] - 2 * std

    up = h.diff()
    down = -low.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=d.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=d.index)
    atr_safe = d["ATR"].replace(0, np.nan)
    plus_di = 100 * plus_dm.ewm(alpha=1 / 14, adjust=False).mean() / atr_safe
    minus_di = 100 * minus_dm.ewm(alpha=1 / 14, adjust=False).mean() / atr_safe
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    d["PLUS_DI"], d["MINUS_DI"] = plus_di, minus_di
    d["ADX"] = dx.ewm(alpha=1 / 14, adjust=False).mean()
    d["VOL20"] = volume.rolling(20).mean()
    d["VOL5"] = volume.rolling(5).mean()
    return d


def technical_score(d: pd.DataFrame) -> tuple[list[dict], float]:
    last = d.iloc[-1]
    price = safe_float(last["Close"])
    rows: list[dict] = []

    def add(metric: str, value: str, points: int, maximum: int, assessment: str) -> None:
        rows.append({"metric": metric, "value": value, "points": points,
                     "max_points": maximum, "assessment": assessment})

    for period, weight in [(20, 1), (50, 1), (200, 2)]:
        avg = safe_float(last.get(f"SMA{period}"))
        if avg is None or price is None:
            continue
        bullish = price > avg
        add(f"Price vs SMA{period}", fmt(avg), weight if bullish else -weight,
            weight, "Above average" if bullish else "Below average")

    rsi = safe_float(last.get("RSI"))
    if rsi is not None:
        if rsi < 30:
            points, note = 1, "Oversold; not an automatic buy"
        elif rsi <= 60:
            points, note = 2, "Healthy momentum zone"
        elif rsi <= 70:
            points, note = 1, "Strong momentum; monitor risk"
        else:
            points, note = -1, "Overbought; pullback risk"
        add("RSI (14)", fmt(rsi, 1), points, 2, note)

    macd, signal = safe_float(last.get("MACD")), safe_float(last.get("MACD_SIGNAL"))
    if macd is not None and signal is not None:
        bullish = macd > signal
        add("MACD", "Bullish" if bullish else "Bearish", 1 if bullish else -1, 1,
            "MACD above signal" if bullish else "MACD below signal")

    adx = safe_float(last.get("ADX"))
    plus_di, minus_di = safe_float(last.get("PLUS_DI")), safe_float(last.get("MINUS_DI"))
    if adx is not None and plus_di is not None and minus_di is not None:
        if adx >= 25:
            points = 1 if plus_di > minus_di else -1
            note = "Trend has strength; direction from DI"
        else:
            points, note = 0, "Trend strength weak / sideways"
        add("ADX (14)", fmt(adx, 1), points, 1, note)

    vol5, vol20 = safe_float(last.get("VOL5")), safe_float(last.get("VOL20"))
    if vol5 is not None and vol20 and vol20 > 0:
        ratio = vol5 / vol20
        add("5-day / 20-day volume", f"{ratio:.2f}x", 1 if ratio > 1.2 else 0, 1,
            "Volume expansion" if ratio > 1.2 else "No strong volume expansion")

    return rows, score_rows(rows)


def strategy_plan(d: pd.DataFrame, config: dict, fundamental_score: float,
                  tech_score: float, market_price: float) -> dict:
    last = d.iloc[-1]
    atr = safe_float(last.get("ATR"))
    ema20 = safe_float(last.get("EMA20"))
    rsi = safe_float(last.get("RSI"))
    if atr is None or atr <= 0 or ema20 is None:
        return {"status": "INSUFFICIENT DATA", "reason": "ATR/EMA indicators unavailable"}

    n_support = min(int(config["support_window"]), len(d))
    lows = pd.to_numeric(d["Low"].tail(n_support), errors="coerce").dropna()
    highs = pd.to_numeric(d["High"].tail(n_support), errors="coerce").dropna()
    if lows.empty or highs.empty:
        return {"status": "INSUFFICIENT DATA", "reason": "Support/resistance unavailable"}
    support = float(lows.min())
    resistance = float(highs.max())

    # A pullback zone is a reference, not a guaranteed fill or a recommendation.
    entry_low = max(0.01, ema20 - 0.5 * atr)
    entry_high = max(entry_low, ema20 + 0.25 * atr)
    breakout = resistance + 0.1 * atr

    # Use a stop below support where sensible, otherwise ATR-based stop.
    proposed_entry = min(max(market_price, entry_low), entry_high)
    if market_price > entry_high:
        proposed_entry = entry_high
    stop_atr = proposed_entry - float(config["atr_mult"]) * atr
    stop_support = support - 0.2 * atr
    stop = max(stop_atr, stop_support) if stop_support < proposed_entry - 0.75 * atr else stop_atr
    stop = min(stop, proposed_entry - 0.1 * atr)
    risk_per_share = proposed_entry - stop

    targets = [proposed_entry + multiple * risk_per_share for multiple in config["rr"]]
    risk_budget = CAPITAL * RISK_PER_TRADE_PCT / 100.0
    max_cash = CAPITAL * MAX_CAPITAL_PER_TRADE_PCT / 100.0
    qty_by_risk = int(risk_budget // risk_per_share) if risk_per_share > 0 else 0
    qty_by_cash = int(max_cash // proposed_entry) if proposed_entry > 0 else 0
    quantity = max(0, min(qty_by_risk, qty_by_cash))
    planned_risk = quantity * risk_per_share
    rr = (targets[0] - proposed_entry) / risk_per_share if risk_per_share > 0 else None

    # A conservative rule-based label; this is not a probability of success.
    trend_ok = market_price > (safe_float(last.get("SMA50")) or -math.inf)
    if fundamental_score >= 65 and tech_score >= 60 and trend_ok and (rsi is None or rsi < 72):
        status = "SETUP FAVOURABLE — verify entry conditions"
    elif tech_score >= 50 and (rsi is None or rsi < 78):
        status = "WATCH / WAIT FOR CONFIRMATION"
    else:
        status = "AVOID NEW ENTRY FOR NOW"

    if market_price > breakout:
        trigger_note = "Price is above the calculated resistance trigger; check for a confirmed close and volume."
    elif market_price >= entry_low and market_price <= entry_high:
        trigger_note = "Price is within the reference pullback zone; confirm trend, liquidity and market conditions."
    else:
        trigger_note = "Wait for the reference pullback zone or a confirmed breakout; do not chase automatically."

    return {
        "status": status, "current_price": market_price,
        "buy_zone_low": entry_low, "buy_zone_high": entry_high,
        "breakout_trigger": breakout, "reference_entry": proposed_entry,
        "stop_loss": stop, "target_1": targets[0], "target_2": targets[1],
        "target_3": targets[2], "support": support, "resistance": resistance,
        "atr": atr, "risk_per_share": risk_per_share,
        "risk_percent": 100 * risk_per_share / proposed_entry if proposed_entry else None,
        "risk_reward_to_target_1": rr, "quantity": quantity,
        "estimated_investment": quantity * proposed_entry,
        "planned_loss_at_stop": planned_risk, "risk_budget": risk_budget,
        "trigger_note": trigger_note,
        "rules": {
            "short_term": "Reassess daily; avoid holding through major events without a plan.",
            "swing": "Review after each daily close; exit if stop is hit or setup fails.",
            "positional": "Review weekly and after quarterly results; do not widen the stop to avoid a loss.",
            "long_term": "Use the levels as risk references only; base investment decisions on business quality, valuation and thesis changes.",
        },
    }


# -------------------------------- NEWS --------------------------------
def fetch_news(ticker: yf.Ticker) -> tuple[list[str], int, str]:
    headlines: list[str] = []
    try:
        for item in (ticker.news or [])[:15]:
            content = item.get("content") or {}
            headline = item.get("title") or content.get("title")
            if headline:
                headlines.append(str(headline))
    except Exception:
        pass

    score = 0
    for headline in headlines:
        words = re.findall(r"[a-z]+", headline.lower())
        score += sum(word in POSITIVE_WORDS for word in words)
        score -= sum(word in NEGATIVE_WORDS for word in words)
    label = "Positive" if score >= 2 else ("Negative" if score <= -2 else "Neutral / inconclusive")
    return headlines, score, label


# -------------------------------- CHART --------------------------------
def save_chart(d: pd.DataFrame, symbol: str, plans: dict) -> str | None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        x = d.tail(220)
        fig, (price_ax, rsi_ax) = plt.subplots(
            2, 1, figsize=(13, 8), sharex=True,
            gridspec_kw={"height_ratios": [3, 1]}
        )
        price_ax.plot(x.index, x["Close"], label="Close", linewidth=1.5)
        price_ax.plot(x.index, x["SMA20"], label="SMA20", linewidth=1)
        price_ax.plot(x.index, x["SMA50"], label="SMA50", linewidth=1)
        if x["SMA200"].notna().any():
            price_ax.plot(x.index, x["SMA200"], label="SMA200", linewidth=1)
        price_ax.set_title(f"{symbol} — price and moving averages")
        price_ax.legend(loc="best", fontsize=8)
        rsi_ax.plot(x.index, x["RSI"], label="RSI (14)")
        rsi_ax.axhline(70, linestyle="--", linewidth=0.8)
        rsi_ax.axhline(30, linestyle="--", linewidth=0.8)
        rsi_ax.set_ylim(0, 100)
        rsi_ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        filename = OUTPUT_DIR / f"{symbol.replace('.', '_')}_analysis.png"
        fig.savefig(filename, dpi=130)
        plt.close(fig)
        return str(filename)
    except Exception as exc:
        print(f"  Chart skipped: {exc}")
        return None


# -------------------------------- MAIN --------------------------------
def analyze(user_input: str) -> dict | None:
    symbol = resolve_symbol(user_input)
    if not symbol:
        print(f"Could not resolve '{user_input}'. Try an exact Yahoo symbol such as RELIANCE.NS or TCS.NS.")
        return None

    ticker = yf.Ticker(symbol)
    print(f"\nLoading market and company data for {symbol} ...")
    info = get_info(ticker)
    currency = str(info.get("currency") or ("INR" if symbol.endswith((".NS", ".BO")) else ""))
    history_by_strategy: dict[str, pd.DataFrame] = {}

    # Use the longest daily history for fundamentals/technical overview.
    try:
        daily = ticker.history(period="2y", interval="1d", auto_adjust=True)
    except Exception:
        daily = pd.DataFrame()
    if daily is None or daily.empty or len(daily) < 40:
        print("Not enough daily price history. Check the symbol or data provider.")
        return None

    daily_indicators = compute_indicators(daily)
    last = daily_indicators.iloc[-1]
    market_price = safe_float(last.get("Close"))
    if market_price is None:
        print("Could not read the latest close price.")
        return None

    try:
        # Latest close may be a previous session close, not a live quote.
        info_price = safe_float(info.get("currentPrice") or info.get("regularMarketPrice"))
    except Exception:
        info_price = None

    section(f"COMPANY OVERVIEW — {info.get('longName') or info.get('shortName') or symbol} ({symbol})")
    print(f"  Sector / industry      : {info.get('sector', 'N/A')} / {info.get('industry', 'N/A')}")
    print(f"  Latest adjusted close  : {fmt(market_price)} {currency}")
    print(f"  Yahoo quote field      : {fmt(info_price)} {currency} (may be delayed)")
    print(f"  Market cap             : {money(info.get('marketCap'), currency)}")
    print(f"  52-week high / low     : {fmt(info.get('fiftyTwoWeekHigh'))} / {fmt(info.get('fiftyTwoWeekLow'))}")
    print(f"  Beta                   : {fmt(info.get('beta'))}")
    print(f"  Promoter/insider stake : {percent(info.get('heldPercentInsiders'))}")
    print(f"  Institutional stake    : {percent(info.get('heldPercentInstitutions'))}")
    print(f"  Analyst recommendation : {info.get('recommendationKey', 'N/A')} "
          f"({info.get('numberOfAnalystOpinions', 'N/A')} opinions)")
    summary = info.get("longBusinessSummary")
    if summary:
        print(f"\n  Business summary: {str(summary)[:700]}")

    f_rows, f_score = analyze_fundamentals(info)
    section(f"FUNDAMENTAL SNAPSHOT — score {f_score}/100 (available Yahoo fields only)")
    print_rows(f_rows)
    trends = print_financial_trends(ticker, currency)

    t_rows, t_score = technical_score(daily_indicators)
    section(f"DAILY TECHNICAL SNAPSHOT — score {t_score}/100")
    print_rows(t_rows)
    support20 = safe_float(daily_indicators["Low"].tail(20).min())
    support60 = safe_float(daily_indicators["Low"].tail(min(60, len(daily_indicators))).min())
    resistance20 = safe_float(daily_indicators["High"].tail(20).max())
    resistance60 = safe_float(daily_indicators["High"].tail(min(60, len(daily_indicators))).max())
    high52 = safe_float(daily_indicators["High"].tail(min(252, len(daily_indicators))).max())
    low52 = safe_float(daily_indicators["Low"].tail(min(252, len(daily_indicators))).min())
    print(f"\n  20-day support/resistance : {fmt(support20)} / {fmt(resistance20)}")
    print(f"  60-day support/resistance : {fmt(support60)} / {fmt(resistance60)}")
    print(f"  Available 52-week high/low: {fmt(high52)} / {fmt(low52)}")
    print(f"  RSI / ATR / ADX           : {fmt(last.get('RSI'))} / {fmt(last.get('ATR'))} / {fmt(last.get('ADX'))}")

    headlines, news_score, news_label = fetch_news(ticker)
    section(f"RECENT NEWS — {news_label} (keyword score {news_score:+d})")
    if headlines:
        for headline in headlines[:10]:
            print(f"  • {headline}")
    else:
        print("  No recent headlines returned by the data provider.")
    print("  Note: headline scoring is a crude keyword count, not a validated sentiment model.")

    plans: dict[str, dict] = {}
    for strategy_name, config in STRATEGIES.items():
        if config["interval"] == "1wk":
            try:
                hist = ticker.history(period=config["period"], interval="1wk", auto_adjust=True)
            except Exception:
                hist = pd.DataFrame()
        else:
            hist = daily.tail(260 if config["period"] in ("1y", "2y") else 130)
        if hist is None or hist.empty or len(hist) < config["min_bars"]:
            plans[strategy_name] = {
                "status": "INSUFFICIENT HISTORY",
                "reason": f"Need about {config['min_bars']} bars; provider returned {0 if hist is None else len(hist)}."
            }
            continue
        indicators = compute_indicators(hist)
        score_rows, strategy_tech_score = technical_score(indicators)
        plans[strategy_name] = strategy_plan(indicators, config, f_score, strategy_tech_score, market_price)
        plans[strategy_name]["technical_score"] = strategy_tech_score

    section("STRATEGY COMPARISON — prices are calculated reference levels, not guaranteed signals")
    for name, plan in plans.items():
        print(f"\n  {name}")
        print(f"    Decision             : {plan.get('status', 'N/A')}")
        if "reference_entry" not in plan:
            print(f"    Reason               : {plan.get('reason', 'Data unavailable')}")
            continue
        print(f"    Current reference px : {fmt(plan['current_price'])} {currency}")
        print(f"    Pullback buy zone    : {fmt(plan['buy_zone_low'])} – {fmt(plan['buy_zone_high'])}")
        print(f"    Breakout trigger     : above {fmt(plan['breakout_trigger'])} with close + volume confirmation")
        print(f"    Reference entry      : {fmt(plan['reference_entry'])}")
        print(f"    Stop loss            : {fmt(plan['stop_loss'])} ({fmt(plan['risk_percent'])} % risk/share)")
        print(f"    Target 1             : {fmt(plan['target_1'])} (R/R {fmt(plan['risk_reward_to_target_1'])})")
        print(f"    Target 2             : {fmt(plan['target_2'])}")
        print(f"    Target 3             : {fmt(plan['target_3'])}")
        print(f"    Support / resistance : {fmt(plan['support'])} / {fmt(plan['resistance'])}")
        print(f"    Suggested quantity   : {plan['quantity']} shares")
        print(f"    Estimated investment: {money(plan['estimated_investment'], currency)}")
        print(f"    Planned stop risk    : {money(plan['planned_loss_at_stop'], currency)}")
        print(f"    Entry note           : {plan['trigger_note']}")

    # Keep data from the run together in one report object.
    result = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "symbol": symbol,
        "company": info.get("longName") or info.get("shortName") or symbol,
        "currency": currency,
        "latest_adjusted_close": market_price,
        "quote_field_price": info_price,
        "fundamental_score": f_score,
        "fundamental_rows": f_rows,
        "financial_trends": trends,
        "technical_score": t_score,
        "technical_rows": t_rows,
        "levels": {
            "support20": support20, "support60": support60,
            "resistance20": resistance20, "resistance60": resistance60,
            "high52": high52, "low52": low52,
        },
        "news": {"score": news_score, "sentiment": news_label, "headlines": headlines},
        "strategies": plans,
        "limitations": [
            "Yahoo Finance data may be delayed, missing or inaccurate.",
            "Scoring thresholds are heuristic and not sector-adjusted.",
            "News sentiment is keyword-based and may misclassify context.",
            "Targets are mathematical scenarios, not forecasts or guarantees.",
            "Transaction costs, taxes, slippage, liquidity and overnight gaps are not modelled.",
        ],
    }

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    chart_path = save_chart(daily_indicators, symbol, plans) if SAVE_CHART else None
    if chart_path:
        print(f"\n  Chart saved to: {chart_path}")
    if SAVE_JSON:
        json_path = OUTPUT_DIR / f"{symbol.replace('.', '_')}_report.json"
        with json_path.open("w", encoding="utf-8") as f:
            json.dump(clean_for_json(result), f, ensure_ascii=False, indent=2, default=str)
        print(f"  JSON report saved to: {json_path}")

    section("FINAL CHECKLIST BEFORE BUYING")
    print("  1. Verify latest price, quarterly results and exchange/company filings.")
    print("  2. Compare valuation and financial ratios with the same-sector peers.")
    print("  3. Check promoter pledging, auditor remarks, related-party dealings and corporate actions.")
    print("  4. Check Nifty/sector trend, upcoming earnings, liquidity and volume.")
    print("  5. Place a stop based on your plan; stop orders can slip during gaps.")
    print("  6. Do not buy solely because a score or target says BUY.")
    print("\n  Educational use only — no guaranteed returns; you can lose money.")
    return result


def main() -> None:
    args = [arg for arg in sys.argv[1:] if not arg.startswith("-") and not arg.endswith(".json")]
    if args and "ipykernel" not in sys.modules:
        analyze(" ".join(args))
        return

    print("A-to-Z Stock Analyzer — enter a company name or ticker; type q to quit.")
    while True:
        try:
            stock = input("\nStock name / ticker: ").strip()
            if stock.lower() in {"q", "quit", "exit"}:
                break
            if not stock:
                continue
            analyze(stock)
        except KeyboardInterrupt:
            print("\nStopped.")
            break
        except Exception as exc:
            print(f"\nAnalysis failed: {exc}")
            print("Check internet access, ticker spelling, Yahoo Finance availability and package versions.")


if __name__ == "__main__":
    main()


A-to-Z Stock Analyzer — enter a company name or ticker; type q to quit.

Stock name / ticker: UPL 

Loading market and company data for UPL.NS ...

 COMPANY OVERVIEW — UPL Limited (UPL.NS)
  Sector / industry      : Basic Materials / Agricultural Inputs
  Latest adjusted close  : 524.70 INR
  Yahoo quote field      : 524.70 INR (may be delayed)
  Market cap             : ₹44,352.06 Cr
  52-week high / low     : 812.20 / 476.45
  Beta                   : 0.54
  Promoter/insider stake : 33.7%
  Institutional stake    : 42.0%
  Analyst recommendation : buy (17 opinions)

  Business summary: UPL Limited, together with its subsidiaries, manufactures and sells pesticides, insecticides, and micronutrients in India, Brazil, the United States, the United Kingdom, and internationally. It operates through Crop protection, Seeds & Post-Harvest solutions, and Non-Agro segments. The company offers herbicides, fungicides, insecticides, acaricides, seed treatment, adjuvants, bio-solutions, public heal